# Biological error analysis

This notebook runs target-cell error analysis: it identifies which non-target cell types
most often absorb prediction errors, measures the neighbor-label composition around target
cells at each downsampling condition, and produces a conservative failure-mode summary
that links marker strength to recovery performance.

The analysis is expensive because it re-runs kNN for every (target, representation, fraction,
seed) condition.  Use `--fractions 0.1 --seeds 0 --max-cells 3000` for a fast debug pass.

Mirrors: `scripts/run_error_analysis.py`

In [ ]:
import subprocess
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
for _p in [PROJECT_ROOT, PROJECT_ROOT / "src"]:
    if str(_p) not in sys.path:
        sys.path.insert(0, str(_p))
assert (PROJECT_ROOT / "src" / "rarecell").exists(), PROJECT_ROOT
PROJECT_ROOT

## Required inputs

| Input | Description |
|---|---|
| `config/benchmark_config.yaml` | Benchmark config: `dataset_path`, `label_column`, `fractions`, `seeds`, `representations`, `k_neighbors` |
| `data/processed/pbmc5k_10x_citeseq_representations.h5ad` | Benchmark-ready AnnData with `obsm` PCA embeddings (produced by `baseline_representations.ipynb`) |
| `results/metrics/rare_cell_benchmark_raw.csv` *(optional)* | Raw benchmark results — used to join marker AUC context into the failure-mode summary |
| `results/tables/marker_auc_by_fraction.csv` *(optional)* | Marker AUC at each fraction — adds biological context to the summary |

The script can reuse existing output CSVs if they are fresh and cover the requested
grid (checked via mtime and condition coverage). Pass `--force` to recompute.

## Canonical script command

```bash
python scripts/run_error_analysis.py --config config/benchmark_config.yaml
```

Debug (fast) command:
```bash
python scripts/run_error_analysis.py --config config/benchmark_config.yaml \
    --fractions 0.1 --seeds 0 --representations rna_pca --max-cells 3000 --verbose
```

Key functions used internally:
- `rarecell.error_analysis.run_error_analysis_grid(adata, ...)` — kNN-based error analysis across the condition grid
- `rarecell.error_analysis.summarize_failure_modes(benchmark_raw, marker_auc, neighbor_composition, target_error)` — conservative failure-mode summary
- `rarecell.plotting.plot_failure_modes(summary, neighbors, errors, output_path)` — failure-mode figure
- `rarecell.plotting.plot_marker_vs_recovery_scatter(summary, output_path)` — marker strength vs. recovery scatter

In [ ]:
result = subprocess.run(
    [
        sys.executable,
        "scripts/run_error_analysis.py",
        "--config", "config/benchmark_config.yaml",
    ],
    cwd=PROJECT_ROOT,
    text=True,
    capture_output=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise SystemExit(result.returncode)

## Expected outputs

| File | Description |
|---|---|
| `results/tables/target_cell_error_analysis.csv` | Which non-target labels most often absorb target-cell prediction errors, by representation and fraction |
| `results/tables/target_neighbor_composition.csv` | Distribution of neighbor labels for target cells at each (fraction, seed, representation) condition |
| `results/tables/biological_interpretation_summary.csv` | Joined marker AUC + recovery + failure-mode label per condition |
| `results/figures/target_neighbor_composition_heatmap.png` | Heatmap of mean neighbor fraction by label and representation |
| `results/figures/target_error_absorption_heatmap.png` | Heatmap of fraction of target errors absorbed by each non-target label |
| `results/figures/marker_vs_recovery_scatter.png` | Scatter of mean marker AUC vs. mean recovery (F1) per condition |
| `results/logs/error_analysis.log` | Run log with timing and condition counts |

In [ ]:
outputs = [
    "results/tables/target_cell_error_analysis.csv",
    "results/tables/target_neighbor_composition.csv",
    "results/tables/biological_interpretation_summary.csv",
    "results/figures/target_neighbor_composition_heatmap.png",
    "results/figures/target_error_absorption_heatmap.png",
    "results/figures/marker_vs_recovery_scatter.png",
]
[(path, (PROJECT_ROOT / path).exists()) for path in outputs]

## Summary

In [ ]:
print("Generated outputs:")
for path in outputs:
    p = PROJECT_ROOT / path
    status = "OK" if p.exists() else "MISSING"
    print(f"  [{status}] {path}")
print()
print("Deviations from run_error_analysis.py:")
print("  - No file-level logging beyond what the script prints to stdout here.")
print("  - The script logs to results/logs/error_analysis.log; this notebook does not.")
print("  - Skipped-condition CSV (results/logs/error_analysis_skipped_conditions.csv)")
print("    is written by the script only when some conditions were skipped.")
print()
print("Note: the full grid (all fractions, seeds, representations) can take >10 min.")
print("For inspection, re-run with --fractions 0.1 --seeds 0 --max-cells 3000.")